# PARC2026 — 74 M3 minimal simulator smoke

Notebook 73 training smoke PASS後、75a/75b本番trainingの前に評価系を確認します。forward/equal-dataのsmoke checkpointだけを使い、`libero_spatial` 10 tasks × 2 evaluation seeds × 3 models = **60 episodes**。promotion evidenceではありません。OpenVLAはJIT merge後にmerged weightsだけcleanupします。fresh Colabでもmodel runtime/LIBERO runtimeを再構築しますが、72a〜72d probeや1800秒benchmarkは再実行しません。


In [ ]:
import json, os, subprocess, traceback
from pathlib import Path
from google.colab import drive, userdata

drive.mount('/content/drive')
if not os.environ.get('HF_TOKEN'):
    try:
        os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    except Exception:
        pass
if not os.environ.get('HF_TOKEN'):
    raise RuntimeError('HF_TOKEN is required; add it to Colab Secrets as HF_TOKEN')
ROOT = Path('/content/parc2026')
ROOT.mkdir(parents=True, exist_ok=True)
REPO = ROOT / 'py_AI_m3_minimal_sim_smoke'
PIN = 'a2f517ed85f4a685e591f1be3b77d59f4a7fdaca'
URL = 'https://github.com/yu37330/py_AI.git'
if not (REPO / '.git').exists():
    if REPO.exists() and any(REPO.iterdir()):
        raise RuntimeError(f'Existing non-Git directory: {REPO}')
    subprocess.run(['git', 'clone', '--no-checkout', URL, str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', PIN], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', '--force', PIN], check=True)
got = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
if got != PIN:
    raise RuntimeError(f'Repository pin mismatch: {got}')
print('74 simulator smoke code:', got, flush=True)
env = os.environ.copy()
env['PY_AI_REPO'] = str(REPO)
env['PARC_ROOT'] = str(ROOT)
env['PARC_DRIVE_ROOT'] = '/content/drive/MyDrive/parc2026-cache'
env['PARC_M3_EXECUTE'] = '1'
DRIVE = Path(env['PARC_DRIVE_ROOT'])
setup_status = DRIVE / 'model-benchmark-v1/m3-simulator-minimal-smoke-v1/runtime-setup/runtime_setup_status.json'
setup_log = DRIVE / 'model-benchmark-v1/m3-simulator-minimal-smoke-v1/runtime-setup/runtime_setup.log'
summary = DRIVE / 'model-benchmark-v1/m3-simulator-minimal-smoke-v1/m3_minimal_simulator_smoke_summary.json'
try:
    subprocess.run([
        'python', '-u', str(REPO / 'tools/colab/prepare_m3_simulator_runtimes.py')
    ], cwd=str(REPO), env=env, check=True)
    subprocess.run([
        'python', '-u', str(REPO / 'tools/colab/run_m3_minimal_simulator_smoke.py')
    ], cwd=str(REPO), env=env, check=True)
except Exception:
    print('=== 74 FAILURE DIAGNOSTICS ===', flush=True)
    for path in (setup_status, setup_log, summary):
        print(f'--- {path} ---', flush=True)
        if path.is_file():
            text = path.read_text(encoding='utf-8', errors='replace')
            print(text[-12000:], flush=True)
        else:
            print('missing', flush=True)
    raise
print('=== 74 COMPLETE ===', flush=True)
print('Minimal simulator smoke only. Benchmark training/promotion/final evaluation has NOT started.', flush=True)
